<h1 align = "center">
Rwanda Maize-Urea Trial
</h1>

<h2 align = "center">
Data Enrichment and Further Analysis
</h2> 

# Objectives

We already know that the soil data came with two variables complteely empty. However, in answering the research question on important drivers of maize yield, it would be a miss to exclude the other variables despite not having been provided.

We want to enrich the data with all possible data from external databases and re-assess the model we had already built in treatment-effect analysis to see whether the additional variables improve the model significantly. 

- We, however, don't just want to fill the missing values with averages or generate plausible-looking numbers. 

### Imports and Setup

In [1]:
import pandas as pd
import requests

In [2]:
from pathlib import Path

# Defining path to folder
data_path = Path(r"../01_Data/processed")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

In [3]:

field_clean = load_csv("field_clean.csv")

# Check coordinate availability
print("Field records:", len(field_clean))
print("Missing latitude:", field_clean["lat"].isna().sum())
print("Missing longitude:", field_clean["long"].isna().sum())

# Summarize coordinate ranges
print(field_clean[["lat", "long"]].describe())

# Count unique field-season combinations
print(
    "Unique field-season combinations:",
    field_clean[["block_id", "season"]].drop_duplicates().shape[0]
)

# Check whether a field-season has conflicting coordinates
coordinate_check = (
    field_clean
    .groupby(["block_id", "season"])
    .agg(
        latitude_count=("lat", "nunique"),
        longitude_count=("long", "nunique")
    )
)

print(
    "Field-season groups with conflicting coordinates:",
    (
        (coordinate_check["latitude_count"] > 1)
        | (coordinate_check["longitude_count"] > 1)
    ).sum()
)

Field records: 399
Missing latitude: 0
Missing longitude: 0
              lat        long
count  399.000000  399.000000
mean    -2.053361   29.754235
std      0.340776    0.419353
min     -2.554515   29.084352
25%     -2.360176   29.473978
50%     -2.074883   29.737257
75%     -1.835101   29.907522
max     -1.429856   30.511245
Unique field-season combinations: 399
Field-season groups with conflicting coordinates: 0


### Enrichment Dataset

We want to prepare the enrichment dataset by creating a clean look-up table for the external data extraction.

- This should help us avoid repeatedly downloading soil properties for every plot as several plots belong to the same field.

In [4]:
field_locations = (
    field_clean[
        ["block_id", "season", "lat", "long"]
    ]
    .drop_duplicates()
    .copy()
)

# Keep only records with usable coordinates
field_locations = field_locations.dropna(
    subset=["lat", "long"]
)

print("Field locations for enrichment:", len(field_locations))
print(field_locations.head())

Field locations for enrichment: 399
                               block_id season       lat       long
0  adfa31ad-95bf-453d-94f1-d163a97756ee    21A -2.021072  30.334707
1  d8807f25-e675-49ae-87a3-d228eca23c72    21A -2.021166  30.334817
2  262efa71-6ff4-4e18-b6f1-0c3d293504c0    21A -2.020386  30.338258
3  95c016dd-908a-4f07-9e49-926e9a9f4f19    21A -2.020589  30.337583
4  1d5c61a0-ca6b-4db8-87c3-5ca46c7cad6d    21A -2.018119  30.338658


We will use the coordinates to extract the external soil estimates, retain the source's units and depth intervals, and then merge the resulting soil table back into the plot dataset using the field identifiers and season.

- We'll start by preparing the field-location lookup

From the coordinate check, we establish that there are 399 field-season records with no missing coordinates and no conflicting coordinates within a field-season.

- These are 399 field locations across the trial seasons, not necessarily 399 unique physical sites

    - If the same field appears in multiple seasons we want to avoid treating its repeated coordinates as separate locations when extracting static soil properties. 

- We want to create one record per unique coordinate pair to avoid repeated soil extraction for fields at exactly the same location

In [5]:
field_locations = (
    field_clean[["block_id", "season", "lat", "long"]]
    .drop_duplicates()
    .copy()
)

field_locations = field_locations.dropna(subset=["lat", "long"])

# One extraction location per unique coordinate pair
unique_locations = (
    field_locations[["lat", "long"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_locations["location_id"] = (
    unique_locations.index + 1
)

print("Field-season records:", len(field_locations))
print("Unique geographic locations:", len(unique_locations))

unique_locations.head()

Field-season records: 399
Unique geographic locations: 291


,lat,long,location_id
0,-2.021072,30.334707,1
1,-2.021166,30.334817,2
2,-2.020386,30.338258,3
3,-2.020589,30.337583,4
4,-2.018119,30.338658,5


#### External Soils Properties Extraction

We'll use SoilGrids. Its gridded maps provide predicted soil pH, nitrogen, organic carbon, sand, silt, clay, and other properties at approximately 250 m resolution, with estimates at multiple depths.

For our analysis, we want to start with five properties at the 0-5 cm and 5-15 cm depths

|Property	|Why it matters for maize yield|
|----|----|
|pH	|Soil acidity and nutrient availability|
|Total nitrogen	|Nutrient availability and soil fertility|
|Organic carbon	|Indicator of soil organic matter and soil condition|
|Sand	|Soil texture and drainage characteristics|
|Clay	|Soil texture, water retention, and nutrient-holding characteristics|



SoilGrids provides these predictions, but the original REST API has had availability issues; the official documentation identifies the Web Coverage Service (WCS) and Google Earth Engine as alternative access routes.

What we want to do first is retrieve a small test sample before extracting values for all locations. 

- This should let us verify access, variable units, depth definitions, and the returned data structure without committing to a lengthy extraction that may fail.

One adjustment to the earlier plan: the official SoilGrids documentation recommends WCS for retrieving subsets of its maps, while its REST API has had availability issues. 

- So For a manageable first pass, we'll test the WCS route for soil pH at 0–5 cm, verify the returned raster and units, and then expand to the other properties.

In [6]:
#Install spatial-data libraries

#%pip install rasterio requests pandas numpy

In [7]:
import requests
import pandas as pd
import numpy as np
import rasterio
from rasterio.warp import transform
from pathlib import Path

#### Test SoilGrids access at one field location

The current SoilGrids documentation also lists a point-query REST endpoint, which is a simpler first test than downloading entire raster files. 

- It can return soil property estimates for a specified coordinate, depth, and prediction statistic

We want to first test one location 

In [14]:
import requests
import json

# Use the first available field location
test_location = unique_locations.iloc[0]

url = "https://isric.org/explore/soilgrids"

params = {
    "lon": float(test_location["long"]),
    "lat": float(test_location["lat"]),
    "property": ["phh2o", "nitrogen", "soc", "sand", "clay", "silt"],
    "depth": ["0-5cm", "5-15cm"],
    "value": ["mean"]
}

try:
    response = requests.get(url, params=params, timeout=60)

    print("HTTP status:", response.status_code)
    print("Request URL:", response.url)

    response.raise_for_status()
    soil_test = response.json()

    print(json.dumps(soil_test, indent=2)[:6000])

except requests.RequestException as error:
    print("SoilGrids request failed:", error)

HTTP status: 200
Request URL: https://isric.org/explore/soilgrids?lon=30.33470688&lat=-2.021071808&property=phh2o&property=nitrogen&property=soc&property=sand&property=clay&property=silt&depth=0-5cm&depth=5-15cm&value=mean
SoilGrids request failed: Expecting value: line 1 column 1 (char 0)


That timeout suggests the SoilGrids service did not respond within 60 seconds. It does not mean our coordinates or dataset are wrong.

Rather than repeatedly retrying the same endpoint, let's switch to a more reliable enrichment workflow. 

We want to obtain documented external data, preserve provenance, and integrate it correctly—not to spend hours troubleshooting one API.

We'll try one alternative access route, and if it is also unavailable, we'll use another source rather than let this block the project.

#### Test alternative SoilGrids services

The official documentation identifies the Web Coverage Service (WCS) as a supported route for retrieving subsets of soil maps

In [9]:
import requests

wcs_url = "https://maps.isric.org/mapserv"

params = {
    "map": "/map/phh2o.map",
    "SERVICE": "WCS",
    "VERSION": "2.0.1",
    "REQUEST": "GetCapabilities"
}

try:
    response = requests.get(
        wcs_url,
        params=params,
        timeout=60
    )

    print("HTTP status:", response.status_code)
    print("Response preview:")
    print(response.text[:1500])

    response.raise_for_status()

    if "CoverageSummary" in response.text:
        print("\nWCS capabilities appear to be available.")
    else:
        print("\nThe service responded, but we need to inspect its response.")

except requests.RequestException as error:
    print("WCS request failed:", error)

WCS request failed: HTTPSConnectionPool(host='maps.isric.org', port=443): Max retries exceeded with url: /mapserv?map=%2Fmap%2Fphh2o.map&SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCapabilities (Caused by ConnectTimeoutError(<HTTPSConnection(host='maps.isric.org', port=443) at 0x25859879310>, 'Connection to maps.isric.org timed out. (connect timeout=60)'))


Both SoilGrids access routes have timed out, so it looks like a service-access issue rather than a problem with our data. 